<a href="https://colab.research.google.com/github/Divyanshu8410/NLP/blob/main/26_Text%20Generation%20using%20GPT-2%20on%20a%20custom%20dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
pip install transformers torch

In [9]:
import torch
from torch.utils.data import Dataset
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)

# 1. Create Custom Dataset File
data = """
Artificial Intelligence is changing the world.
Machine learning helps computers learn from data.
Deep learning is a part of machine learning.
Natural Language Processing helps computers understand human language.
GPT-2 is a language generation model.
Python is widely used in artificial intelligence.
"""

with open("custom_data.txt", "w", encoding="utf-8") as file:
    file.write(data.strip())

# 2. Load GPT-2 Tokenizer and Model
print("Loading GPT-2 model...")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained("gpt2")
model.config.pad_token_id = tokenizer.pad_token_id

# 3. Define a modern Custom PyTorch Dataset to replace the deprecated TextDataset
class CustomTextDataset(Dataset):
    def __init__(self, file_path, tokenizer, block_size=32):
        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()

        tokenized_text = tokenizer(
            text,
            truncation=True,
            max_length=1024,
            return_tensors=None
        )["input_ids"]

        self.examples = []
        for i in range(0, len(tokenized_text) - block_size + 1, block_size):
            self.examples.append(tokenized_text[i : i + block_size])

        # If the text is shorter than block_size, pad or use the whole block
        if not self.examples and len(tokenized_text) > 0:
            self.examples.append(tokenized_text)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        return torch.tensor(self.examples[i], dtype=torch.long)

# Prepare Custom Dataset
train_dataset = CustomTextDataset(
    file_path="custom_data.txt",
    tokenizer=tokenizer,
    block_size=32
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# 4. Training Arguments
training_args = TrainingArguments(
    output_dir="./gpt2_custom",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    save_steps=10,
    logging_steps=5,
    report_to="none"
)

# 5. Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator
)

# 6. Fine-Tune GPT-2
print("\nTraining GPT-2 on custom dataset...")
trainer.train()

# 7. Generate New Text
prompt = "Artificial Intelligence"
inputs = tokenizer(prompt, return_tensors="pt")

model.eval()
with torch.no_grad():
    output = model.generate(
        inputs["input_ids"],
        max_length=60,
        num_return_sequences=1,
        do_sample=True,
        temperature=0.8,
        top_k=50,
        pad_token_id=tokenizer.eos_token_id
    )

generated_text = tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

# 8. Display Result
print("\nInput Prompt:", prompt)
print("\nGenerated Text:")
print(generated_text)

# 9. Save Fine-Tuned Model
trainer.save_model("./gpt2_custom")
tokenizer.save_pretrained("./gpt2_custom")

print("\nModel saved successfully!")

Loading GPT-2 model...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Training GPT-2 on custom dataset...


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Input Prompt: Artificial Intelligence

Generated Text:
Artificial Intelligence: A Tool for Understanding and Improving Machine Learning, Springer-Verlag, 2010

Cognitive Psychology: A Multidisciplinary Approach to Machine Learning, Springer-Verlag, 2010

Machine Learning and Artificial Intelligence: A Multidisciplinary Approach to Machine Learning, Springer-


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved successfully!
